# The same conversations on a hosted agent

Deploys `newsletter-hosted-agent`, a hosted agent that runs its own code, and repeats the flow of
[08-13-01](08-13-01-start-edition-conversations.ipynb) and [08-13-02](08-13-02-return-and-resume.ipynb) (prompt agent variant)
against it. The conversation calls are the same as for the prompt agent.
Background: [`08-13-00-newsletter-conversations.md`](08-13-00-newsletter-conversations.md).

**Prerequisites:** `uv sync`, `az login`, and `ALPHA_FOUNDRY_ACCOUNT`, `ALPHA_FOUNDRY_PROJECT_ENDPOINT`,
`ALPHA_FOUNDRY_CORE_CONNECTION` and `CHAT_MODEL` in `.env` (set by the 05 labs). Your identity needs
Foundry Project Manager on the project, and Owner or User Access Administrator on the spoke resource group
for the role assignment in Step 3.

## Step 1: Configuration

Load `.env` and connect to the Team Alpha project.

In [1]:
import hashlib, json, os, re, subprocess, time, zipfile
from pathlib import Path
from dotenv import load_dotenv
import openai
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import (
    AgentEndpointProtocol, CodeConfiguration, HostedAgentDefinition, ProtocolVersionRecord,
)

repo_root = Path(subprocess.run(
    'git rev-parse --show-toplevel', shell=True, capture_output=True, text=True
).stdout.strip())
load_dotenv(repo_root / '.env', override=True)

# Suffix - first 6 chars of SHA-256 of subscription ID (matches the project spoke naming)
SUBSCRIPTION_ID  = subprocess.run('az account show --query id -o tsv', shell=True, capture_output=True, text=True).stdout.strip()
SUFFIX           = hashlib.sha256((SUBSCRIPTION_ID + 'v2').encode()).hexdigest()[:6]
SPOKE_RG         = f"rg-foundry-spoke-alpha-{SUFFIX}"
ACCOUNT_NAME     = os.environ["ALPHA_FOUNDRY_ACCOUNT"]
PROJECT_ENDPOINT = os.environ["ALPHA_FOUNDRY_PROJECT_ENDPOINT"]
MODEL            = f'{os.environ["ALPHA_FOUNDRY_CORE_CONNECTION"]}/{os.environ["CHAT_MODEL"]}'
AGENT_NAME       = "newsletter-hosted-agent"
AGENT_DIR        = Path("newsletter-hosted-agent")   # main.py and requirements.txt
EDITIONS_DIR     = repo_root / "assets" / "contoso-daily-brief-dataset" / "editions"

client = AIProjectClient(endpoint=PROJECT_ENDPOINT, credential=DefaultAzureCredential(), allow_preview=True)

print(f"Project : {PROJECT_ENDPOINT}")
print(f"Agent   : {AGENT_NAME}")
print(f"Model   : {MODEL}")

Project : https://aif-spoke-alpha-c2676f.services.ai.azure.com/api/projects/project-alpha-c2676f
Agent   : newsletter-hosted-agent
Model   : core-alpha/gpt-5.4-mini


## Step 2: Deploy the agent from source

Zip [`newsletter-hosted-agent/main.py`](newsletter-hosted-agent/main.py) with its `requirements.txt`, register
it as a hosted agent version and wait for the service to build it. The code is an Agent Framework agent with
no tools. An unchanged zip returns the existing version, so a re-run does not rebuild.

In [2]:
zip_path = Path("build") / f"{AGENT_NAME}.zip"
zip_path.parent.mkdir(exist_ok=True)
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for name in ("main.py", "requirements.txt"):
        # A fixed timestamp keeps the zip's hash stable. The mode marks a regular file, which the service requires.
        info = zipfile.ZipInfo(name, date_time=(1980, 1, 1, 0, 0, 0))
        info.compress_type = zipfile.ZIP_DEFLATED
        info.external_attr = 0o100644 << 16
        z.writestr(info, (AGENT_DIR / name).read_bytes())

with zip_path.open("rb") as code:
    version = client.agents.create_version_from_code(
        agent_name=AGENT_NAME,
        definition=HostedAgentDefinition(
            protocol_versions=[ProtocolVersionRecord(protocol=AgentEndpointProtocol.RESPONSES, version="2.0.0")],
            cpu="1", memory="2Gi",
            code_configuration=CodeConfiguration(
                runtime="python_3_13", entry_point=["python", "main.py"], dependency_resolution="remote_build"),
            environment_variables={"PROJECT_ENDPOINT": PROJECT_ENDPOINT, "CHAT_MODEL": MODEL},
        ),
        code=code,
    )

t0 = time.time()
while (status := client.agents.get_version(agent_name=AGENT_NAME, agent_version=version.version).status) == "creating":
    assert time.time() - t0 < 1200, "the version is still creating after 20 minutes"
    time.sleep(10)

print(f"{AGENT_NAME} version {version.version}: {getattr(status, 'value', status)} after {time.time() - t0:.0f}s")
assert status == "active"
agent_client = client.get_openai_client(agent_name=AGENT_NAME)

newsletter-hosted-agent version 1: active after 51s


## Step 3: Let the agent call the model

A hosted agent runs as its own identity. Grant that identity Foundry User on the project so its code can
call the model, and wait for a new assignment to apply.

In [3]:
principal_id = client.agents.get_version(agent_name=AGENT_NAME, agent_version=version.version).instance_identity.principal_id
project_id = (
    f"/subscriptions/{SUBSCRIPTION_ID}/resourceGroups/{SPOKE_RG}/providers/Microsoft.CognitiveServices"
    f"/accounts/{ACCOUNT_NAME}/projects/{PROJECT_ENDPOINT.rstrip('/').split('/')[-1]}"
)
FOUNDRY_USER_ROLE_ID = "53ca6127-db72-4b80-b1b0-d745d6d5456d"

def az(command):
    r = subprocess.run(f"az role assignment {command}", shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(r.stderr.strip()[:300])
    return r.stdout.strip()

assignment_ids = f"list --scope '{project_id}' --role {FOUNDRY_USER_ROLE_ID} --query \"[?principalId=='{principal_id}'].id\" -o tsv"
if az(assignment_ids):
    print("Foundry User on the project: already granted")
else:
    az(f"create --assignee-object-id {principal_id} --assignee-principal-type ServicePrincipal "
       f"--role {FOUNDRY_USER_ROLE_ID} --scope '{project_id}'")
    print("Foundry User on the project: granted, waiting 90s for it to apply")
    time.sleep(90)

Foundry User on the project: granted, waiting 90s for it to apply


## Step 4: Create one conversation per edition

The same calls as 08-13-01: the edition id, date and title as metadata, and the edition text as the first
item. An edition that already has a conversation is skipped.

In [4]:
editions = [json.loads(path.read_text(encoding="utf-8")) for path in sorted(EDITIONS_DIR.glob("ed-*.json"))]

def edition_text(edition):
    """An edition as plain text, one block per article, each starting with the article id."""
    articles = "\n\n".join(
        f'[{a["article_id"]}] {a["category"]}: {a["headline"]}\n{a["summary"]}' for a in edition["articles"])
    return (f'Contoso Daily Brief, edition {edition["edition_id"]}, published {edition["published"]}\n'
            f'{edition["title"]}\n\n{articles}')

def list_conversations():
    """The conversations on this agent's endpoint, one row each: `id` (the Foundry conversation id), `created_at`
    and `metadata`. The openai SDK has no list method, so this is a raw GET."""
    return agent_client.get("/conversations", cast_to=dict, options={"params": {"limit": 100}})["data"]

foundry_conv_ids = {c["metadata"].get("edition_id"): c["id"] for c in list_conversations()}
for e in editions:
    if e["edition_id"] in foundry_conv_ids:
        print(f'{e["edition_id"]}  exists   {foundry_conv_ids[e["edition_id"]]}')
        continue
    conversation = agent_client.conversations.create(
        metadata={"edition_id": e["edition_id"], "edition_date": e["published"], "edition_title": e["title"]},
        items=[{"type": "message", "role": "developer", "content": edition_text(e)}],
    )
    foundry_conv_ids[e["edition_id"]] = conversation.id
    print(f'{e["edition_id"]}  created  {conversation.id}')

ed-001  created  conv_0b7c0747770fba7d00ITG3K0PO0dXhT82FKdTGfxvNyKsx45dr
ed-002  created  conv_07b49a6bd4f6c04500twNQjAlecZp3VZiGnqx5wNj38RWERkGb


ed-003  created  conv_0d0c20bc62e33b9d00gi44FCPjMpEDUfoqFWhgjb9w8HhBYHtT
ed-004  created  conv_0c54ed3f95073d32005zkdYpaFiiFhAsgbKIcvSFcUSxGTf4dR


ed-005  created  conv_093bfa4ef17c3c3e00lt8t77ajUnn4o2cxxXdsaVMnOkIqE6O3


## Step 5: Ask a first question about one edition

The call passes only the conversation id. The first call starts the agent's sandbox, so it takes longer and
is retried while the sandbox is not ready. The question is skipped if a previous run already asked it.

In [5]:
def ask(foundry_conv_id, question, retries=6):
    """One turn in a conversation, retried while the sandbox starts (424) or returns a 5xx."""
    for attempt in range(1, retries + 1):
        try:
            response = agent_client.responses.create(conversation=foundry_conv_id, input=question)
            break
        except openai.APIStatusError as e:
            if e.status_code not in (424, 500, 502, 503) or attempt == retries:
                raise
            print(f"  HTTP {e.status_code}, retrying in 20s ({attempt}/{retries})")
            time.sleep(20)
    assert response.status == "completed", response.error
    return response

EDITION_ID = "ed-001"
foundry_conv_id = foundry_conv_ids[EDITION_ID]

if any(item.role == "user" for item in agent_client.conversations.items.list(foundry_conv_id, limit=100).data):
    print("The first question was asked in an earlier run.")
else:
    t0 = time.time()
    response = ask(foundry_conv_id, f"My pick from this edition is {EDITION_ID}-02. In two sentences, why does it matter?")
    print(f"{response.output_text}\n\n{time.time() - t0:.0f}s, including the sandbox start")

It matters because Relecloud’s liquid-cooled server hall is already turning waste heat into warmth for 2,400 homes, showing a practical way for data centers to benefit local communities [ed-001-02]. It also used 38% less electricity in the pilot than the nearby air-cooled hall, suggesting a meaningful efficiency gain that could scale if the company converts more halls in 2027 [ed-001-02].

8s, including the sandbox start


## Step 6: Find the conversation and read its history back

As in 08-13-02: list the conversations, match the edition id in the metadata, and list the items. The
history is stored by Foundry, not by the agent's sandbox.

In [6]:
conversation = next(c for c in list_conversations() if c["metadata"]["edition_id"] == EDITION_ID)
foundry_conv_id = conversation["id"]
print(f'{foundry_conv_id}\n{json.dumps(conversation["metadata"], indent=2)}\n')

items = agent_client.conversations.items.list(foundry_conv_id, order="asc", limit=100).data
for item in items:
    text = item.content[0].text
    if item.role == "developer":
        lines = text.splitlines()
        text = "\n           ".join(lines[:1] + [line for line in lines if line.startswith("[ed-")])
    print(f"{item.role:>9}: {text}\n")

conv_0b7c0747770fba7d00ITG3K0PO0dXhT82FKdTGfxvNyKsx45dr
{
  "edition_id": "ed-001",
  "edition_date": "2026-11-16",
  "edition_title": "Northwind buys into a port, Relecloud heats homes"
}

developer: Contoso Daily Brief, edition ed-001, published 2026-11-16
           [ed-001-01] Business: Northwind Traders buys 40% of the Alderbrook container terminal
           [ed-001-02] Technology: Relecloud's liquid-cooled server hall now heats 2,400 homes
           [ed-001-03] Science: Trey Research finds a kelp forest recovering twice as fast as predicted
           [ed-001-04] Sport: Alpine Ski House wins the opening relay of the Tailspin Cup

     user: My pick from this edition is ed-001-02. In two sentences, why does it matter?

assistant: It matters because Relecloud’s liquid-cooled server hall is already turning waste heat into warmth for 2,400 homes, showing a practical way for data centers to benefit local communities [ed-001-02]. It also used 38% less electricity in the pilot than th

## Step 7: Resume the conversation

Ask a question that only the stored history can answer. The cell fails if the answer does not name the
article picked in Step 5.

In [7]:
first_question = next(item for item in items if item.role == "user").content[0].text
picked_article_id = re.search(r"ed-\d{3}-\d{2}", first_question).group()

response = ask(foundry_conv_id, "Which article did I pick when we first talked about this edition? Give its article id and headline.")
print(response.output_text)

assert picked_article_id in response.output_text, f"the answer does not name {picked_article_id}"

You picked **ed-001-02**: **Technology: Relecloud's liquid-cooled server hall now heats 2,400 homes** [ed-001-02].


## Cleanup

Leave everything in place to run this notebook again on a later day. To remove the conversations, the role
assignment and the agent with its sessions, uncomment the lines below.

In [8]:
# for c in list_conversations():
#     agent_client.conversations.delete(c["id"])
# az(f"delete --ids {az(assignment_ids)}")
# client.agents.delete(agent_name=AGENT_NAME, force=True)
# print(f"Deleted the conversations, the role assignment and {AGENT_NAME}")